<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/01_Data_Scientist/beginner/06_sql_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL in Python — SQLite & Pandas Integration

As a data scientist you'll query databases constantly. This notebook shows SQL running inside Python via SQLite (no database server needed).

**Topics:** sqlite3, pd.read_sql, DuckDB for DataFrames, 15 progressive SQL exercises

In [ ]:
import sqlite3
import numpy as np
import pandas as pd

np.random.seed(42)

# Create in-memory database
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row  # dict-like access

def sql(query, conn=conn):
    """Helper: run SQL and return a DataFrame."""
    return pd.read_sql_query(query, conn)

# Create tables
conn.executescript("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT, age INTEGER, city TEXT,
    segment TEXT, join_date TEXT
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    name TEXT, category TEXT, price REAL, cost REAL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER, product_id INTEGER,
    quantity INTEGER, order_date TEXT, status TEXT
);
""")

# Insert data
n_customers = 200
cities = ['New York', 'LA', 'Chicago', 'Houston', 'Phoenix']
segments = ['Premium', 'Standard', 'Basic']

customers_data = [
    (i, f'Customer_{i}', np.random.randint(18,70),
     np.random.choice(cities), np.random.choice(segments),
     f'202{np.random.randint(0,4)}-{np.random.randint(1,13):02d}-{np.random.randint(1,28):02d}')
    for i in range(1, n_customers+1)
]
conn.executemany('INSERT INTO customers VALUES (?,?,?,?,?,?)', customers_data)

categories = ['Electronics', 'Clothing', 'Books', 'Food', 'Sports']
products_data = [
    (i, f'Product_{i}', np.random.choice(categories),
     round(np.random.uniform(10, 500), 2),
     round(np.random.uniform(5, 200), 2))
    for i in range(1, 51)
]
conn.executemany('INSERT INTO products VALUES (?,?,?,?,?)', products_data)

n_orders = 2000
statuses = ['completed', 'pending', 'returned', 'cancelled']
orders_data = [
    (i, np.random.randint(1, n_customers+1), np.random.randint(1, 51),
     np.random.randint(1, 10),
     f'202{np.random.randint(2,5)}-{np.random.randint(1,13):02d}-{np.random.randint(1,28):02d}',
     np.random.choice(statuses, p=[0.7, 0.1, 0.1, 0.1]))
    for i in range(1, n_orders+1)
]
conn.executemany('INSERT INTO orders VALUES (?,?,?,?,?,?)', orders_data)
conn.commit()
print('Database created with customers, products, orders tables')

## Basic Queries

In [ ]:
# Q1: Top 5 cities by customer count
print('Q1: Top 5 cities by customer count')
print(sql("""
    SELECT city, COUNT(*) as customer_count
    FROM customers
    GROUP BY city
    ORDER BY customer_count DESC
    LIMIT 5
"""))

# Q2: Revenue by product category (completed orders only)
print('\nQ2: Revenue by category (completed orders)')
print(sql("""
    SELECT p.category,
           COUNT(*) as n_orders,
           SUM(o.quantity * p.price) as total_revenue,
           ROUND(AVG(o.quantity * p.price), 2) as avg_order_value
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    WHERE o.status = 'completed'
    GROUP BY p.category
    ORDER BY total_revenue DESC
"""))

# Q3: Customers with no orders (LEFT JOIN + IS NULL)
print('\nQ3: Customers who never placed an order')
print(sql("""
    SELECT c.customer_id, c.name, c.segment
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.order_id IS NULL
    LIMIT 5
"""))

## Intermediate: CTEs & Window Functions

In [ ]:
# Q4: Top customer per segment by revenue
print('Q4: Top customer per segment (window function)')
print(sql("""
    WITH customer_revenue AS (
        SELECT c.customer_id, c.name, c.segment,
               COALESCE(SUM(o.quantity * p.price), 0) as revenue
        FROM customers c
        LEFT JOIN orders o ON c.customer_id = o.customer_id AND o.status = 'completed'
        LEFT JOIN products p ON o.product_id = p.product_id
        GROUP BY c.customer_id, c.name, c.segment
    ),
    ranked AS (
        SELECT *,
               RANK() OVER (PARTITION BY segment ORDER BY revenue DESC) as rnk
        FROM customer_revenue
    )
    SELECT segment, name, ROUND(revenue, 2) as revenue
    FROM ranked
    WHERE rnk = 1
    ORDER BY revenue DESC
"""))

# Q5: Month-over-month order growth
print('\nQ5: Month-over-month order growth')
print(sql("""
    WITH monthly AS (
        SELECT STRFTIME('%Y-%m', order_date) as month,
               COUNT(*) as orders,
               SUM(quantity * (SELECT price FROM products p WHERE p.product_id = o.product_id)) as revenue
        FROM orders o
        WHERE status = 'completed'
        GROUP BY month
    )
    SELECT month, orders,
           LAG(orders) OVER (ORDER BY month) as prev_month_orders,
           ROUND((orders - LAG(orders) OVER (ORDER BY month)) * 100.0
                 / LAG(orders) OVER (ORDER BY month), 1) as growth_pct
    FROM monthly
    ORDER BY month
    LIMIT 10
"""))

In [ ]:
# Q6: Customer RFM analysis (Recency, Frequency, Monetary)
print('Q6: RFM Analysis — Customer Segmentation via SQL')
rfm = sql("""
    WITH customer_metrics AS (
        SELECT o.customer_id,
               JULIANDAY('now') - JULIANDAY(MAX(order_date)) as recency_days,
               COUNT(*) as frequency,
               SUM(o.quantity * p.price) as monetary
        FROM orders o
        JOIN products p ON o.product_id = p.product_id
        WHERE o.status = 'completed'
        GROUP BY o.customer_id
    )
    SELECT customer_id,
           ROUND(recency_days) as recency_days,
           frequency,
           ROUND(monetary, 2) as monetary,
           NTILE(4) OVER (ORDER BY recency_days ASC)  as R_score,
           NTILE(4) OVER (ORDER BY frequency DESC)    as F_score,
           NTILE(4) OVER (ORDER BY monetary DESC)     as M_score
    FROM customer_metrics
    ORDER BY monetary DESC
    LIMIT 10
""")
print(rfm)
print('\nRFM scores 1=best, 4=worst for R (recency); 4=best for F,M')

## SQL + Pandas Integration

In [ ]:
# Pull SQL result directly into pandas for further analysis
category_df = sql("""
    SELECT p.category, o.status,
           COUNT(*) as orders,
           SUM(o.quantity * p.price) as revenue
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY p.category, o.status
""")

# Pivot in pandas
pivot = category_df.pivot_table(
    values='orders', index='category', columns='status', aggfunc='sum', fill_value=0
)
pivot['return_rate'] = (pivot.get('returned', 0) / pivot.sum(axis=1) * 100).round(1)
print('Orders by Category and Status:')
print(pivot)

# Write pandas DataFrame back to SQLite
import matplotlib.pyplot as plt
pivot['return_rate'].sort_values().plot(kind='barh', color='darkorange', alpha=0.8)
plt.title('Return Rate by Product Category')
plt.xlabel('Return Rate %')
plt.tight_layout(); plt.show()

print('\nTip: For analytics on DataFrames, try DuckDB:')
print('  import duckdb')
print('  duckdb.sql("SELECT category, AVG(revenue) FROM category_df GROUP BY category").df()')